# Crivo: continuar o trabalho e inspecionar os pesos próprios

Os treinamentos desta rodada foram executados pelo Codex. **Você não precisa repetir o treino ou contratar uma GPU.** Este notebook carrega os candidatos já treinados: interpretação de fatos com nomes normalizados e, separadamente, geração de texto. Ambos são experimentais, sem aprovação automática para o chat.

A normalização e a gramática de cópia ajudam a rede; seus resultados não medem conversa livre. O gerador recebeu um piloto pequeno de redação, que também precisa de avaliação semântica. Veja abaixo os resultados e falhas conservados. Só arquitetura, tokenizer e pesos próprios do Crivo; Torch, NumPy e tokenizers são infraestrutura.

In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/CRIVO-associacao')
BRANCH='codex/associacao-fatos-20261008'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/HROSONE/CRIVO.git',str(REPO)])
else:
    b=subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
    if b!=BRANCH:raise RuntimeError('A pasta pertence a outra branch; use uma pasta nova.')
print('Revisão:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
ASSOC=REPO/'experimentos/associacao_fatos_20261008'
GERACAO=REPO/'experimentos/geracao_dialogo_20261008'


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.2,<3','numpy>=1.24,<3','tokenizers>=0.20,<1'])


## O que aconteceu nesta rodada

Os quatro resultados de interpretação distinguem texto bruto, normalização e ajuste de pesos. O relatório de geração mostra perda e respostas reais; menor perda ou resposta terminada não certificam coerência. Nenhum dos candidatos altera automaticamente o Crivo do chat.

In [ ]:
r=json.loads((ASSOC/'avaliacao_ablation/resumo.json').read_text())
for nome,medidas in r['resultados'].items():
    print(nome,'painel novo:',json.dumps(medidas['novo']['limitado'],ensure_ascii=False))
print('Critérios locais:',r['criterios'])
print('Geração:',json.loads((GERACAO/'avaliacao/resumo.json').read_text()))
print('Leitura semântica:',json.loads((GERACAO/'avaliacao/leitura_manual.json').read_text()))


## Inspecionar a associação de fatos

Edite as falas. A proposta aponta para fontes originais, mesmo quando os nomes são normalizados. Uma operação executável ainda pode usar fatos errados. A normalização suporta até oito nomes candidatos com inicial maiúscula, em uma gramática limitada; não representa IDs persistentes. Este exemplo guiado não vale como teste independente.

In [ ]:
FALAS=['O plano Azul custa 23 reais. Verde: preço de 35 reais. Compare Azul e Verde.',
       'Corrigindo Azul: o custo real é 41 reais. Compare Azul e Verde.']
p=Path('/content/entrada-associacao.json');p.write_text(json.dumps(FALAS,ensure_ascii=False))
subprocess.check_call([sys.executable,str(ASSOC/'inspecionar.py'),'--normalizado',
    '--pesos',str(ASSOC/'candidato_normalizado/pesos.pt'),'--entrada',str(p)])


## Inspecionar a geração de respostas

Esta célula usa o gerador causal próprio de 2,61M, separado do intérprete de 2,95M. Edite as falas para inspecionar continuidade. O histórico usa as respostas geradas, não respostas de referência. Este piloto tem poucos dados e pode produzir respostas incoerentes; veja as falhas completas. Históricos longos podem perder turnos para caber em 256 tokens.

As entradas dessas duas células ficam somente nos arquivos temporários do Colab; não entram no treino nem são copiadas automaticamente ao Drive.

In [ ]:
FALAS_DIALOGO=['Hoje fiquei alguns minutos olhando a luz entrar pela janela. Foi uma pausa boa.',
                'Não estou pedindo uma rotina. Quero só conversar sobre essa sensação.']
p=Path('/content/entrada-geracao.json');p.write_text(json.dumps(FALAS_DIALOGO,ensure_ascii=False))
subprocess.check_call([sys.executable,str(GERACAO/'inspecionar.py'),
    '--modelo',str(GERACAO/'candidato'),'--entrada',str(p)])


## Treinamento

Os pesos acima já estão treinados. Repetir os mesmos dados não corrige as limitações observadas. O trabalho seguinte depende dos erros e de mais diálogos variados com fontes e partições conservadas. Este notebook não inicia treinamentos, consome quota de GPU ou promete aprovação. Os scripts publicados permitem reprodução em pastas novas e retomada a partir dos checkpoints completos de uma execução própria. Os checkpoints de Adam/RNG desta rodada permanecem no laboratório local; os pesos escolhidos e relatórios foram publicados.

O Codex executou a rodada em CPU. O notebook foi conferido localmente; ainda não foi executado numa sessão real do Colab.